In [1]:
import pandas as pd

In [2]:
df_injured = pd.read_csv("accidentes-injury-histo.csv")

/tmp/ipykernel_5983/2128208587.py:1: DtypeWarning: Columns (0: case_id_pkey, 1: juris) have mixed types. Specify dtype option on import or set low_memory=False.
  df_injured = pd.read_csv("accidentes-injury-histo.csv")


In [3]:
df_injured.info()

<class 'pandas.DataFrame'>
RangeIndex: 64694 entries, 0 to 64693
Data columns (total 58 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   unique_id                64694 non-null  int64  
 1   cnn_intrsctn_fkey        64657 non-null  float64
 2   cnn_sgmt_fkey            28914 non-null  float64
 3   case_id_pkey             64694 non-null  object 
 4   tb_latitude              64513 non-null  str    
 5   tb_longitude             64513 non-null  str    
 6   geocode_source           64694 non-null  str    
 7   geocode_location         64694 non-null  str    
 8   collision_datetime       64694 non-null  str    
 9   collision_date           64694 non-null  str    
 10  collision_time           64632 non-null  str    
 11  accident_year            64694 non-null  int64  
 12  month                    64694 non-null  str    
 13  day_of_week              64685 non-null  str    
 14  time_cat                 64641 no

### 1. Columnas de localización.
1. Point es identico a tb_latitude y tb_longitude. *Sería la columna a usar en una visualización en mapa.*
2. geocode_source y geocode_location son irrelevantes, la primeraespecifica qué agente es al fuente geográfico, la segunda tiene un único valor.
3. El resto de coumnas son relevantes y nos hablan tanto de en qué calle a ocurrido el accidente, cómo poder dibujar la dirección o el final del accidente si fue mi grabe. Puede ser interesante la columna de "distancia recorrida".

In [4]:
geo_c = ['tb_latitude', 'tb_longitude', 'geocode_source', 'geocode_location', 'juris', 'primary_rd', 'secondary_rd', 'distance', 'direction', 'intersection', 'street_view', 'analysis_neighborhood', 'point']
df_injured.loc[:, geo_c].sample(3)

,tb_latitude,tb_longitude,geocode_source,geocode_location,juris,primary_rd,secondary_rd,distance,direction,intersection,street_view,analysis_neighborhood,point
55867,"37,73972349946","-122,46699101433",SFPD-CROSSROADS,CITY STREET,3801,VICENTE ST,WEST PORTAL AVE,0.0,Not Stated,Intersection <= 20ft,https://maps.google.com/maps?q=&layer=c&cbll=3...,West of Twin Peaks,POINT (-122.466991014 37.739723499)
25488,"37,77886068722","-122,40827158835",SFPD-INTERIM DB,CITY STREET,3801,HOWARD ST,RUSS ST,0.0,Not Stated,Intersection <= 20ft,https://maps.google.com/maps?q=&layer=c&cbll=3...,South of Market,POINT (-122.408271588 37.778860687)
30649,"37,79854807694","-122,42401657242",SFPD-CROSSROADS,CITY STREET,3801,UNION ST,VAN NESS AVE,10.0,West,Intersection <= 20ft,https://maps.google.com/maps?q=&layer=c&cbll=3...,Marina,POINT (-122.424016572 37.798548077)


**geocode_location** *Eliminar*, unicamente hay una categoría.

In [5]:
df_injured.geocode_location.unique()

<StringArray>
['CITY STREET']
Length: 1, dtype: str

Los valores nulos de distancias los imputamos a 0.

In [6]:
df_injured.distance = df_injured.distance.fillna(0)

In [7]:
df_injured.drop(columns=['tb_latitude', 'tb_longitude', 'geocode_location', 'geocode_source', 'street_view', 'secondary_rd'], inplace=True)

### 2. Columnas de temporales.
1. Esta tabla nos da mucha granularidad a la hora de poder mostrar el momento del accidente. Para el modelo no será útil pero para el BI sí.
2. **Imputamos vlaores nulos de horas y días basandonos en la coluna `collision_datetime`**
3. Eliminamos las que repiten información

In [17]:
time_c = ['collision_datetime', 'collision_date', 'collision_time', 'accident_year', 'month', 'day_of_week', 'time_cat']
df_injured.loc[:, time_c].sample(3)

,collision_datetime,collision_date,collision_time,accident_year,month,day_of_week,time_cat
32752,2007-10-22 13:25:00,2007 October 22,13:25:00,2007,October,Monday,10:01 am to 2:00 pm
34196,2024-12-04 10:03:00,2024 December 04,10:03:00,2024,December,Wednesday,10:01 am to 2:00 pm
42133,2015-01-18 22:55:00,2015 January 18,22:55:00,2015,January,Sunday,10:01 pm to 2:00 am


In [9]:
df_injured.loc[:, time_c].info()

<class 'pandas.DataFrame'>
RangeIndex: 64694 entries, 0 to 64693
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   collision_datetime  64694 non-null  str  
 1   collision_date      64694 non-null  str  
 2   collision_time      64632 non-null  str  
 3   accident_year       64694 non-null  int64
 4   month               64694 non-null  str  
 5   day_of_week         64685 non-null  str  
 6   time_cat            64641 non-null  str  
dtypes: int64(1), str(6)
memory usage: 3.5 MB


In [15]:
df_injured.loc[:, time_c].isnull().sum()

collision_datetime     0
collision_date         0
collision_time         0
accident_year          0
month                  0
day_of_week            0
time_cat              53
dtype: int64

In [11]:
df_injured["collision_datetime"] = pd.to_datetime(df_injured["collision_datetime"])

mask_day = df_injured["day_of_week"].isnull()
df_injured.loc[mask_day, "day_of_week"] = df_injured.loc[mask_day, "collision_datetime"].dt.day_name()

mask_time = df_injured["collision_time"].isnull()
df_injured.loc[mask_time, "collision_time"] = df_injured.loc[mask_time, "collision_datetime"].dt.strftime("%H:%M:%S")

In [9]:
df_injured.drop(columns=['time_cat', 'collision_datetime'], inplace=True)

### 3. Columnas de policía.
1. Información sobre los ajentes que reportan y sus comisarias.
2. **Mantenemos superviso_district para la visualización BI**

In [10]:
police_c = ['officer_id', 'reporting_district', 'beat_number', 'police_district', 'control_device']
df_injured.loc[:, police_c].sample(3)

,officer_id,reporting_district,beat_number,police_district,control_device
22717,1375,COI,3116D,TARAVAL,NaN
53302,1279,Bayview,3C2,BAYVIEW,Functioning
63087,845,SOUTHERN,4K3F,SOUTHERN,Functioning


In [11]:
df_injured.drop(columns=police_c, inplace=True)

**Supervisor_district**
1. Conservamos este campo para la inteligencia de negocio.
2. Imputamos los nulos a la moda.
3. Convertimos a String

In [12]:
df_injured.supervisor_district.isnull().sum()

np.int64(262)

In [13]:
# supervisor_district -> str
df_injured['supervisor_district'] = df_injured['supervisor_district'].fillna(df_injured['supervisor_district'].mode()[0])
df_injured['supervisor_district'] = df_injured['supervisor_district'].astype('str')

### 4. Columnas de colisión.
1. Aportan gran cantidad de información sobre el sentido del accidente. Naturaleza de este, causas y partes implicadas.
2. Columnas muy importantes para un modelo de sinisestralidad. Columnas importantísimas cómo: *mviw* y *ped_action*.

In [14]:
colision_c = ['collision_severity', 'type_of_collision', 'mviw', 'ped_action', 'number_killed', 'number_injured']
df_injured.loc[:, colision_c].sample(3)

,collision_severity,type_of_collision,mviw,ped_action,number_killed,number_injured
49714,Injury (Complaint of Pain),Sideswipe,Other Motor Vehicle,No Pedestrian Involved,0.0,2
58222,Injury (Complaint of Pain),Broadside,Other Motor Vehicle,No Pedestrian Involved,0.0,1
10847,Injury (Other Visible),Other,Bicycle,No Pedestrian Involved,0.0,1


In [15]:
df_injured.collision_severity.value_counts()

collision_severity
Injury (Complaint of Pain)    41129
Injury (Other Visible)        18388
Injury (Severe)                4561
Fatal                           615
Medical                           1
Name: count, dtype: int64

In [16]:
df_injured.type_of_collision.value_counts()

type_of_collision
Broadside             19832
Vehicle/Pedestrian    13389
Rear End              10463
Sideswipe              8553
Head-On                3942
Other                  3391
Hit Object             2465
Not Stated             1507
Overturned             1152
Name: count, dtype: int64

In [17]:
df_injured.mviw.value_counts()

mviw
Other Motor Vehicle               29666
Pedestrian                        15076
Bicycle                            8752
Fixed Object                       2895
Parked Motor Vehicle               2806
Non-Collision                      1531
Not Stated                         1510
Other Object                       1309
Motor Vehicle on Other Roadway      973
Train                               147
Animal                               29
Name: count, dtype: int64

In [18]:
df_injured.ped_action.value_counts()

ped_action
No Pedestrian Involved                       48373
Crossing in Crosswalk at Intersection         9263
Crossing Not in Crosswalk                     3242
In Road, Including Shoulder                   1982
Not in Road                                    869
Not Stated                                     718
Crossing in Crosswalk Not at Intersection      222
Approaching/Leaving School Bus                  14
Not In Road                                     11
Name: count, dtype: int64

### 5. Columnas de carretera.
1. Columans realacionas con el entorno del accidente. **Relación con PCI ??**
1. weather_2 es redundante respecto al 1, no aporta mucho matiz. Al igual que el tiempo, reoad_cond_2 es redundante.

In [19]:
road_c = ['road_surface', 'road_cond_1', 'road_cond_2', 'lighting', 'weather_1', 'weather_2']
df_injured.loc[:, road_c].sample(3)

,road_surface,road_cond_1,road_cond_2,lighting,weather_1,weather_2
48313,Dry,No Unusual Condition,Not Stated,Dusk - Dawn,Clear,Not Stated
50436,Dry,No Unusual Condition,Not Stated,Daylight,Clear,Not Stated
31028,Dry,No Unusual Condition,Not Stated,Daylight,Clear,Not Stated


In [20]:
df_injured.road_cond_1.value_counts()

road_cond_1
No Unusual Condition           61125
Not Stated                      1550
Other                            780
Construction or Repair Zone      520
Holes, Deep Ruts                 268
Obstruction on Roadway           147
Loose Material on Roadway        134
Holes, Deep Rut                   97
Reduced Roadway Width             54
Flooded                           19
Name: count, dtype: int64

In [21]:
df_injured.drop(columns=['weather_2', 'road_cond_2'], inplace=True)

### 6. Columnas de legalidad.
1. Columans relacionadas sobre qué infracción se a cometido y quien tine la culpa en el parte del seguro **(presupongo ??)**
2. Puede aportar información al modelo, sobre todo la infracción y el tipo de vehículo implicado. **Información repetida respecto a las columnas del accidente.**
3. Sobra muchas columnas. Columnas interesantes: *vz_pcf_description*, *party1_move_pre_acc* y quizas *dph_col_grp_description*
4. Todas la columnas vz se son repetitivas, unicamente necesitamos la descripción.

In [22]:
law_c = ['vz_pcf_code', 'vz_pcf_group', 'vz_pcf_description', 'vz_pcf_link', 'dph_col_grp', 'dph_col_grp_description', 'party_at_fault', 'party1_type', 'party1_dir_of_travel', 'party1_move_pre_acc', 'party2_type', 'party2_dir_of_travel', 'party2_move_pre_acc']
df_injured.loc[:, law_c].sample(3)

,vz_pcf_code,vz_pcf_group,vz_pcf_description,vz_pcf_link,dph_col_grp,dph_col_grp_description,party_at_fault,party1_type,party1_dir_of_travel,party1_move_pre_acc,party2_type,party2_dir_of_travel,party2_move_pre_acc
41603,22107,22107,Unsafe turn or lane change prohibited,http://leginfo.legislature.ca.gov/faces/codes_...,AA,Vehicle(s) Only Involved,1.0,Driver,West,Making Right Turn,Driver,West,Proceeding Straight
17112,21801(a),"21801(a,b)",Violation of right-of-way - left turn,http://leginfo.legislature.ca.gov/faces/codes_...,AA,Vehicle(s) Only Involved,1.0,Driver,South,Making Left Turn,Driver,North,Proceeding Straight
24082,21800(a),21800(a-c),Violation of right-of-way or uncontrolled inte...,http://leginfo.legislature.ca.gov/faces/codes_...,CC,Vehicle-Bicycle,1.0,Driver,North,Merging,Bicyclist,North,Changing Lanes


In [23]:
df_injured.party1_type.value_counts()

party1_type
Driver            52826
Bicyclist          5234
Pedestrian         5022
Other              1190
Parked Vehicle      381
Not Stated           27
Bicycle               1
Name: count, dtype: int64

In [24]:
df_injured.party1_move_pre_acc.value_counts()

party1_move_pre_acc
Proceeding Straight                       33543
Making Left Turn                          11275
Making Right Turn                          4083
Changing Lanes                             2212
Other                                      1918
Entering Traffic                           1722
Not Stated                                 1388
Backing                                    1384
Stopped In Road                            1210
Making U Turn                              1202
Parked                                      780
Slowing/Stopping                            698
Passing Other Vehicle                       625
Stopped                                     551
Traveling Wrong Way                         488
Ran Off Road                                423
Parking Maneuver                            359
Other Unsafe Turning                        336
Crossed Into Opposing Lane                  226
Merging                                     182
Crossed Into Opposin

In [25]:
df_injured.dph_col_grp_description.value_counts()

dph_col_grp_description
Vehicle(s) Only Involved                    38291
Vehicle-Pedestrian                          15512
Vehicle-Bicycle                              8655
Bicycle Only                                 1188
Bicycle-Pedestrian                            512
Bicycle-Parked Car                            456
Pedestrian Only or Pedestrian-Parked Car       34
Unknown/Not Stated                             19
Vehicle-Bicycle-Pedestrian                     19
Bicycle-Unknown/Not Stated                      6
Name: count, dtype: int64

In [26]:
df_injured.drop(columns=['vz_pcf_code', 'vz_pcf_group', 'vz_pcf_link', 'dph_col_grp', 'party_at_fault'], inplace=True)

**Finalmente quedan unas columnas sobre las actualizaciones del dataset que no importan para nuestro problema.**

In [27]:
metadata_c = ['data_as_of', 'data_updated_at', 'data_loaded_at']
df_injured.loc[:, metadata_c].sample(3)

,data_as_of,data_updated_at,data_loaded_at
44649,2016 Mar 24 12:00:00 AM,2023 Apr 26 12:00:00 AM,2026 May 01 12:27:24 PM
23623,2014 Apr 08 12:00:00 AM,2025 Apr 29 12:00:00 AM,2026 May 01 12:27:24 PM
54686,2017 Dec 26 12:00:00 AM,2023 Apr 26 12:00:00 AM,2026 May 01 12:27:24 PM


In [28]:
df_injured.drop(columns=metadata_c, inplace=True)

In [29]:
df_injured.isnull().sum()

unique_id                      0
cnn_intrsctn_fkey             37
cnn_sgmt_fkey              35780
case_id_pkey                   0
collision_date                 0
collision_time                62
accident_year                  0
month                          0
day_of_week                    9
juris                          1
primary_rd                     1
distance                       0
direction                      1
weather_1                      0
collision_severity             0
type_of_collision              0
mviw                           0
ped_action                     0
road_surface                   0
road_cond_1                    0
lighting                       0
intersection                   1
vz_pcf_description             0
number_killed                  3
number_injured                 0
dph_col_grp_description        2
party1_type                   13
party1_dir_of_travel          12
party1_move_pre_acc           12
party2_type                 4703
party2_dir

In [30]:
#df_injured.to_csv("heridos_limpio.csv",index=False)

In [33]:
df_injured.loc[:, ['cnn_intrsctn_fkey','cnn_sgmt_fkey']]

,cnn_intrsctn_fkey,cnn_sgmt_fkey
0,26037000.0,NaN
1,30731000.0,NaN
2,25186000.0,NaN
3,24378000.0,588000.0
4,30749000.0,8756101.0
...,...,...
64689,25835000.0,NaN
64690,23282000.0,NaN
64691,26552000.0,NaN
64692,27378000.0,8359201.0
